<a href="https://colab.research.google.com/github/LegoKam/ZEIT8025-Project-report/blob/main/analyser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Initialise

In [1]:
!pip install -q pefile requests beautifulsoup4 scikit-learn pandas joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 2.3 MB/s eta 0:00:00


## Download Malware for analysis

In [2]:
import math
import os
import hashlib
import requests
from collections import Counter

import pefile


def download_to_colab_and_sha256(url: str, dest_path: str | None = None) -> str:
    """Download a file into Colab and return its SHA256 hex digest."""
    if dest_path is None:
        name = os.path.basename(url.split("?", 1)[0]) or "downloaded_file"
        dest_path = os.path.join("/content", name)

    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk:
                    f.write(chunk)

    sha256 = hashlib.sha256()
    with open(dest_path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            sha256.update(block)

    digest = sha256.hexdigest()
    print(f"Saved to: {dest_path}")
    print(f"SHA256:   {digest}")
    return digest


def result_label(is_suspicious: bool) -> str:
    return "Suspicious" if is_suspicious else "Benign"


def print_step_result(step: int, title: str, result: str, details=None):
    print(f"\n{'='*60}")
    print(f"Step {step}: {title}")
    print(f"Result: {result}")
    if details:
        for k, v in details.items():
            print(f"  {k}: {v}")
    print(f"{'='*60}")


def shannon_entropy(data: bytes) -> float:
    """Entropy of a binary blob."""
    if not data:
        return 0.0
    counts = Counter(data)
    n = len(data)
    return -sum((c / n) * math.log2(c / n) for c in counts.values())


def _normalize_section_name(raw) -> str:
    name = raw.decode() if isinstance(raw, bytes) else raw
    return name.strip("\x00").lstrip(".").lower()


def compute_section_entropies(path: str) -> dict:
    """Shannon entropy"""
    entropies = {}
    try:
        pe = pefile.PE(path, fast_load=True)
        for section in pe.sections:
            name = _normalize_section_name(section.Name)
            try:
                data = section.get_data()
            except Exception:
                data = b""
            entropies[name or "unnamed"] = round(shannon_entropy(data), 4)
        pe.close()
    except Exception as exc:
        print(f"  [warn] entropy error: {exc}")
    return entropies


In [3]:
from google.colab import userdata

VT_API_KEY = (userdata.get("VT_API_KEY"))

print("Libraries ready.")
print(f"VT_API_KEY: {VT_API_KEY}")

Libraries ready.
VT_API_KEY: c931dbd5c782d08f7e0f45c7b63fb6af5648037ec5734d77b4091a25ca771ce5


## STEP 1 — SHA256 hash of MALWARE_FILE


In [4]:
MALWARE_URL = (
    "https://pocstorage1705.blob.core.windows.net/storage/input/payload/malware1.exe"
    "?sp=r&st=2026-06-11T06:16:27Z&se=2027-06-11T14:31:27Z"
    "&spr=https&sv=2026-02-06&sr=b&sig=ZJhN29TWAWHDSQhpXq9e0%2BmuNyAZJUZn13F3glkgUyA%3D"
)
MALWARE_FILE = "/content/malware1.exe"

file_sha256 = download_to_colab_and_sha256(MALWARE_URL, MALWARE_FILE)

VT_SUSPICIOUS_RATIO = 0.05

vt_details = {"sha256": file_sha256, "malicious": 0, "total_engines": 0, "detection_ratio": 0.0}
step1_result = "Benign"

if VT_API_KEY:
    vt_url = f"https://www.virustotal.com/api/v3/files/{file_sha256}"
    headers = {"x-apikey": VT_API_KEY}
    resp = requests.get(vt_url, headers=headers, timeout=60)
    if resp.status_code == 200:
        stats = resp.json()["data"]["attributes"]["last_analysis_stats"]
        malicious = stats.get("malicious", 0) + stats.get("suspicious", 0)
        total = sum(stats.values())
        ratio = malicious / total if total else 0.0
        vt_details.update({"malicious": malicious, "total_engines": total, "detection_ratio": round(ratio, 4)})
        step1_result = result_label(ratio >= VT_SUSPICIOUS_RATIO or malicious > 0)
    elif resp.status_code == 404:
        vt_details["note"] = "Hash not found in VirusTotal"
    else:
        vt_details["note"] = f"VT API error {resp.status_code}: {resp.text[:200]}"
else:
    vt_details["note"] = "No VT_API_KEY — set it in Colab Secrets"

print_step_result(1, "VirusTotal", step1_result, vt_details)


Saved to: /content/malware1.exe
SHA256:   21a0205f1f68690b400644651f59f750a3852a03f6fbfe0606b14d217fe3a5f8

Step 1: VirusTotal
Result: Suspicious
  sha256: 21a0205f1f68690b400644651f59f750a3852a03f6fbfe0606b14d217fe3a5f8
  malicious: 56
  total_engines: 76
  detection_ratio: 0.7368


## STEP 2 — Section entropy

Shannon entropy per PE section

In [5]:
ENTROPY_SUSPICIOUS_THRESHOLD = 7.0

section_entropies = compute_section_entropies(MALWARE_FILE)
max_entropy = max(section_entropies.values()) if section_entropies else 0.0
avg_entropy = (sum(section_entropies.values()) / len(section_entropies)) if section_entropies else 0.0
high_entropy_sections = [n for n, e in section_entropies.items() if e >= ENTROPY_SUSPICIOUS_THRESHOLD]

step2_result = result_label(
    max_entropy >= ENTROPY_SUSPICIOUS_THRESHOLD
    or avg_entropy >= (ENTROPY_SUSPICIOUS_THRESHOLD - 0.5)
)

print_step_result(2, "Section Entropy", step2_result, {
    "max_entropy": max_entropy,
    "avg_entropy": round(avg_entropy, 4),
    "high_entropy_sections": high_entropy_sections or ["none"],
    "all_sections": section_entropies,
})



Step 2: Section Entropy
Result: Benign
  max_entropy: 6.3092
  avg_entropy: 3.9805
  high_entropy_sections: ['none']
  all_sections: {'text': 6.3092, 'rdata': 4.8245, 'data': 0.8077}
